In [ ]:
# from pathlib import Path
# import hashlib, json, datetime

# def render(template_path: str, **kwargs) -> str:
#     text = Path(template_path).read_text(encoding="utf-8")
#     for k, v in kwargs.items():
#         text = text.replace("{{" + k + "}}", str(v))
#     return text

# prompt = render("prompts/stat_test_select.j2", data_desc=desc, alpha=0.05)
# rec = {
#     "template": "prompts/stat_test_select.j2",
#     "template_sha256": hashlib.sha256(Path("prompts/stat_test_select.j2")
#                                       .read_bytes()).hexdigest(),
#     "rendered_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
#     "rendered_text": prompt,
#     "model_snapshot": "claude-sonnet-4-5@2026-01-01",
#     "temperature": 0.0,
#     "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
# }
# json.dump(rec, open("runs/2026-01-01T10-00Z/prompt.json", "w"),
#           ensure_ascii=False, indent=2)

In [ ]:
# import numpy as np
# def variance_components(Y):           # Y shape = (M, R)
#     M, R = Y.shape
#     mbar = Y.mean(axis=1)
#     gbar = Y.mean()
#     MS_W = ((Y - mbar[:, None]) ** 2).sum() / (M * (R - 1))
#     MS_B = R * ((mbar - gbar) ** 2).sum() / (M - 1)
#     s2e = MS_W
#     s2b = max((MS_B - MS_W) / R, 0.0)   # 截断，避免负值
#     return {"sigma_b": np.sqrt(s2b), "sigma_e": np.sqrt(s2e),
#             "icc": s2b / (s2b + s2e) if (s2b + s2e) > 0 else 0.0}

In [1]:
import numpy as np

rng = np.random.default_rng(0)
n, p = 200, 4
x1 = rng.standard_normal(n)
x2 = rng.standard_normal(n)
x3 = x1 + x2 + 0.0003 * rng.standard_normal(n)   # 近似共线，抬高条件数
x4 = rng.standard_normal(n)
X0 = np.column_stack([x1, x2, x3, x4])
y = X0 @ np.array([1.0, 1.0, -1.0, 0.5]) + 0.5 * rng.standard_normal(n)

A = X0.T @ X0
print(f"cond(XtX) = {np.linalg.cond(A):.3e}")
se3 = np.sqrt(np.linalg.inv(A)[2, 2] * 0.25)     # beta_3 抽样标准误（近似）
print(f"beta_3 的抽样标准误(近似) = {se3:.3e}")

def xtx_blocked(X, K):
    # 模拟 K 路并行归约：块内 XtX，块和顺序累加
    T = np.zeros((X.shape[1], X.shape[1]))
    for c in np.array_split(X, K):
        T += c.T @ c
    return T

def xty_blocked(X, yv, K):
    t = np.zeros(X.shape[1])
    for cX, cy in zip(np.array_split(X, K), np.array_split(yv, K)):
        t += cX.T @ cy
    return t

ref_beta = np.linalg.solve(X0.T @ X0, X0.T @ y)
Ks = [1, 2, 4, 8, 16, 32]              # 环境：分块数（线程数的代理）
M, R = len(Ks), 5
Y = np.empty((M, R))
for i, K in enumerate(Ks):
    for r in range(R):                 # 运行：数据载入顺序（种子）
        perm = np.random.default_rng(1000 + 10 * i + r).permutation(n)
        Xp, yp = X0[perm], y[perm]
        b = np.linalg.solve(xtx_blocked(Xp, K), xty_blocked(Xp, yp, K))
        Y[i, r] = b[2] - ref_beta[2]

print("beta_3 偏差 (x1e-6):")
print(np.round(Y * 1e6, 3))

mbar = Y.mean(axis=1)
MS_W = ((Y - mbar[:, None]) ** 2).sum() / (M * (R - 1))
MS_B = R * ((mbar - Y.mean()) ** 2).sum() / (M - 1)
s2e = MS_W; s2b = max((MS_B - MS_W) / R, 0.0)
icc = s2b / (s2b + s2e) if s2b + s2e > 0 else 0.0
print(f"sigma_b={np.sqrt(s2b):.3e}  sigma_e={np.sqrt(s2e):.3e}  ICC={icc:.3f}")

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
cond(XtX) = 9.481e+07
beta_3 的抽样标准误(近似) = 1.185e+02
beta_3 偏差 (x1e-6):
[[-0.15  -0.302  0.051 -0.151 -0.101]
 [-0.403 -0.252 -0.605 -0.302  0.001]
 [-0.151 -0.302 -0.252 -0.504 -0.555]
 [-0.353 -0.353 -0.101 -0.605 -0.252]
 [-0.504 -0.454 -0.706 -0.806 -0.353]
 [-0.454 -0.605 -0.505 -0.352 -0.605]]
sigma_b=1.335e-07  sigma_e=1.704e-07  ICC=0.380


In [2]:
import numpy as np

rng = np.random.default_rng(0)
n = 200_000
x = 1000.0 + rng.standard_normal(n)

def var_twopass_f64(a):                # 参照实现：float64 两遍公式
    m = a.mean()
    return float(((a - m) ** 2).mean())

def var_naive_f32(a, K):
    # 一遍公式 sum(x^2)/n - mean^2，float32 累加，K 路分块
    a32 = a.astype(np.float32)
    s1 = np.float32(0.0); s2 = np.float32(0.0)
    for c in np.array_split(a32, K):
        s1 += np.float32(c.sum(dtype=np.float32))
        s2 += np.float32((c * c).sum(dtype=np.float32))
    m = float(s1) / len(a)
    return float(s2) / len(a) - m * m

ref = var_twopass_f64(x)
print(f"参照实现 (float64 两遍): {ref:.6f}")

bs = np.array([var_twopass_f64(x[np.random.default_rng(r).integers(0, n, n)])
               for r in range(20)])
print(f"抽样噪声参照: bootstrap SE = {bs.std(ddof=1):.4f}")

Ks = [1, 2, 4, 8, 16, 32]
M, R = len(Ks), 5
Y = np.empty((M, R))
for i, K in enumerate(Ks):             # 环境：分块数
    for r in range(R):                 # 运行：数据载入顺序
        perm = np.random.default_rng(1000 + 10 * i + r).permutation(x)
        Y[i, r] = var_naive_f32(perm, K) - ref

print("naive float32 一遍公式的偏差:")
print(np.round(Y, 4))

mbar = Y.mean(axis=1)
MS_W = ((Y - mbar[:, None]) ** 2).sum() / (M * (R - 1))
MS_B = R * ((mbar - Y.mean()) ** 2).sum() / (M - 1)
s2e = MS_W; s2b = max((MS_B - MS_W) / R, 0.0)
icc = s2b / (s2b + s2e) if s2b + s2e > 0 else 0.0
print(f"sigma_b={np.sqrt(s2b):.4f}  sigma_e={np.sqrt(s2e):.4f}  ICC={icc:.3f}")

参照实现 (float64 两遍): 1.002401
抽样噪声参照: bootstrap SE = 0.0027
naive float32 一遍公式的偏差:
[[ 0.0498  0.0459  0.0498 -0.0322 -0.036 ]
 [-0.1141  0.1278 -0.0322 -0.1141  0.2059]
 [ 0.0459 -0.0322 -0.0322 -0.1141 -0.1141]
 [-0.0322  0.1278  0.0459 -0.0322 -0.1141]
 [ 0.0459  0.2098  0.2059 -0.0322 -0.0322]
 [ 0.2098  0.2878 -0.1102  0.3698  0.0536]]
sigma_b=0.0504  sigma_e=0.1209  ICC=0.148


In [ ]:
# import json, hashlib, subprocess, datetime, os, pathlib

# def git_commit():
#     try:
#         return subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
#     except Exception:
#         return None

# def sha256_file(p): return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()

# run_dir = pathlib.Path("runs") / datetime.datetime.now(
#     datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
# run_dir.mkdir(parents=True, exist_ok=True)

# params = {"seed": 20260101, "lr": 1e-3, "epochs": 50, "model": "resnet18"}
# json.dump({"params": params, "git_commit": git_commit(),
#            "data_sha256": {p: sha256_file(p) for p in ["data/train.parquet"]}},
#           open(run_dir / "params.json", "w"), indent=2)

# # ... 训练，得到 metrics 与 artifacts ...
# metrics = {"auc": 0.873, "auc_se": 0.012, "n_boot": 200}
# json.dump(metrics, open(run_dir / "metrics.json", "w"), indent=2)
# json.dump({p.name: sha256_file(p) for p in pathlib.Path("results").iterdir()},
#           open(run_dir / "artifacts.json", "w"), indent=2)


In [ ]:
# import json, pathlib

# BASE = pathlib.Path("runs_demo")
# log = BASE / "verification_log.jsonl"

# # ---- 1. 写入：每次验证一条记录，失败与成功同等保留 ----
# records = [
#     {"ts": "2026-09-20T09:12:00Z", "run_id": "20260920T091200Z", "check": "numeric",
#      "status": "fail", "detail": "绝对路径 C:\\Users\\a\\data 硬编码于 fit.py 第 41 行",
#      "inputs": {"fit.py": "3f2a..."}},
#     {"ts": "2026-09-21T10:02:00Z", "run_id": "20260921T100200Z", "check": "numeric",
#      "status": "fail", "detail": "种子未固定，两次运行 AUC 差 0.021",
#      "inputs": {"fit.py": "8b1c..."}},
#     {"ts": "2026-09-22T15:40:00Z", "run_id": "20260922T154000Z", "check": "numeric",
#      "status": "fail", "detail": "环境漂移：容器外重跑偏差 3e-4，超出容差 1e-5",
#      "inputs": {"fit.py": "8b1c..."}},
#     {"ts": "2026-09-23T11:05:00Z", "run_id": "20260923T110500Z", "check": "numeric",
#      "status": "pass", "detail": "在镜像 sha256:9a1f 下重跑，与归档结果差 2e-9",
#      "inputs": {"fit.py": "8b1c...", "input.csv": "c4d9..."}},
#     {"ts": "2026-09-24T14:30:00Z", "run_id": "20260924T143000Z", "check": "human_review",
#      "status": "pass", "detail": "复核人 L.W.，对照 codebook 逐字段核对",
#      "inputs": {"review.md": "e05b..."}},
# ]
# BASE.mkdir(parents=True, exist_ok=True)
# with open(log, "w", encoding="utf-8") as f:
#     for rec in records:
#         f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# # ---- 2. 读取器：汇总通过率、首次通过前的失败次数、失败明细 ----
# def summarize(log_path):
#     recs = [json.loads(l) for l in
#             pathlib.Path(log_path).read_text(encoding="utf-8").splitlines() if l.strip()]
#     n_pass = sum(r["status"] == "pass" for r in recs)
#     first_pass = next((i for i, r in enumerate(recs) if r["status"] == "pass"), None)
#     fails = [r for r in recs if r["status"] == "fail"]
#     print(f"记录总数 {len(recs)}，通过 {n_pass}，失败 {len(fails)}")
#     if first_pass is not None:
#         print(f"首次通过前有 {first_pass} 次失败尝试（多重比较校正时须计入）")
#     for r in fails:
#         print(f"  [FAIL] {r['ts']} {r['check']}: {r['detail']}")
#     # 检查项覆盖：机器可检验证据之外是否有人工复核
#     kinds = {r["check"] for r in recs}
#     if "human_review" not in kinds:
#         print("提示: 仅有机器可检验证据，缺少人工复核记录（17.5.1）")

# summarize(log)

In [ ]:
# import hashlib, json, pathlib, subprocess, sys

# def sha256(p):
#     h = hashlib.sha256()
#     with open(p, "rb") as f:
#         for b in iter(lambda: f.read(1 << 20), b""):
#             h.update(b)
#     return h.hexdigest()

# def check(manifest_path="runs/manifest.json", code_files=("src",)):
#     m = json.load(open(manifest_path))
#     problems = []

#     # 6/23：数据哈希与产物哈希是否仍然匹配
#     for role in ("inputs", "artifacts"):
#         for path, want in m.get(role, {}).items():
#             if not pathlib.Path(path).exists():
#                 problems.append(f"[缺失] {role}: {path}")
#             elif sha256(path) != want:
#                 problems.append(f"[哈希不一致] {role}: {path}")

#     # 11：代码 commit 是否与 manifest 记录一致
#     head = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
#     if head != m.get("git_commit"):
#         problems.append(f"[commit 不一致] 记录={m.get('git_commit')} 当前={head}")

#     # 13：是否存在硬编码的绝对路径或疑似密钥
#     for d in code_files:
#         for f in pathlib.Path(d).rglob("*.py"):
#             t = f.read_text(encoding="utf-8", errors="ignore")
#             if "/home/" in t or "C:\\\\Users" in t:
#                 problems.append(f"[硬编码路径] {f}")
#             if "sk-" in t.lower():
#                 problems.append(f"[疑似密钥] {f}")

#     print("PASS: 无问题" if not problems else "\n".join(problems))
#     return 0 if not problems else 1

# sys.exit(check())

In [3]:
import hashlib, json, pathlib

BASE = pathlib.Path("runs_demo")

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

# ---- 1. 构造一次运行的输入与产物 ----
run = BASE / "runs" / "20260925T081500Z"; run.mkdir(parents=True)
(run / "input.csv").write_text("x,y\n1,2\n3,4\n5,7\n", encoding="utf-8")
metrics = {"estimate": 0.8731, "se": 0.0122, "n": 12480}
(run / "metrics.json").write_text(json.dumps(metrics), encoding="utf-8")

# ---- 2. 写 manifest：把全部输入与产物哈希绑定 ----
manifest = {
    "run_id": run.name,
    "created_utc": "2026-09-25T08:15:00Z",
    "git_commit": None,
    "command": "python fit.py --data input.csv",
    "inputs":    {"input.csv":    sha256_file(run / "input.csv")},
    "artifacts": {"metrics.json": sha256_file(run / "metrics.json")},
}
mp = run / "manifest.json"
mp.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

# ---- 3. 校验函数：逐文件重算哈希并与 manifest 比对 ----
def verify(run_dir):
    m = json.loads((pathlib.Path(run_dir) / "manifest.json").read_text(encoding="utf-8"))
    ok = True
    for role in ("inputs", "artifacts"):
        for name, want in m[role].items():
            p = pathlib.Path(run_dir) / name
            if not p.exists():
                print(f"FAIL  {role}/{name}: 文件缺失"); ok = False
            else:
                got = sha256_file(p)
                print(f"{'PASS' if got == want else 'FAIL'}  {role}/{name}")
                ok &= (got == want)
    return ok

print("== 首次校验 ==")
assert verify(run)

# ---- 4. 篡改演示：改一个数字后校验失败 ----
metrics["estimate"] = 0.8732
(run / "metrics.json").write_text(json.dumps(metrics), encoding="utf-8")
print("== 篡改后校验 ==")
assert not verify(run)

# ---- 5. 运行间链：下一 run 的 manifest 记录上一 run 的 manifest 哈希 ----
prev_sha = sha256_file(mp)
print(f"链头: manifest.json sha256 = {prev_sha[:16]}...（写入下一个 run 的 manifest.parent 字段）")

== 首次校验 ==
PASS  inputs/input.csv
PASS  artifacts/metrics.json
== 篡改后校验 ==
PASS  inputs/input.csv
FAIL  artifacts/metrics.json
链头: manifest.json sha256 = aae3405b81786ee7...（写入下一个 run 的 manifest.parent 字段）
